# Spark Sandbox

Abre una SparkSession contra el clúster standalone. Ejecuta primero `python lessons/00_seed.py`. Usa las celdas marcadas como experimentos para cambiar particiones, filtros y estrategias de join.

## 0. Sesión y ejecutores

In [ ]:
from pyspark.sql import functions as F
from lab.config import RAW, session
from lab.spark import describe, executor_memory_status

spark = session('sandbox', delta=True)
describe(spark)
executor_memory_status(spark)

## 1. Datos y consulta SQL

In [ ]:
schema = 'meter_id string, ts timestamp, kwh double, voltage double'
readings = spark.read.option('header', True).schema(schema).csv(f'file://{RAW}/readings')
meters = spark.read.option('header', True).option('inferSchema', True).csv(f'file://{RAW}/meters.csv')
readings.createOrReplaceTempView('readings')
meters.createOrReplaceTempView('meters')
spark.sql('SELECT count(*) AS rows, count(DISTINCT meter_id) AS meters FROM readings').show()

### Juega aquí: una agregación por ciudad
Cambia el filtro, las columnas y la agrupación. ¿Qué operaciones generan shuffle?

In [ ]:
result = spark.sql('''
SELECT city, customer_type, round(sum(kwh) / 1000, 1) AS mwh
FROM readings JOIN meters USING (meter_id)
WHERE kwh >= 0
GROUP BY city, customer_type
ORDER BY mwh DESC
''')
result.show(12, truncate=False)
result.explain(mode='formatted')

## 2. Repartition, AQE y joins

In [ ]:
print('input partitions:', readings.rdd.getNumPartitions())
print('repartition(16):', readings.repartition(16).rdd.getNumPartitions())
print('coalesce(4):', readings.coalesce(4).rdd.getNumPartitions())
print('AQE:', spark.conf.get('spark.sql.adaptive.enabled'))

### Juega aquí: broadcast vs sort-merge
Desactiva broadcast con umbral `-1`; compara los planes y el shuffle en la Spark UI.

In [ ]:
spark.conf.set('spark.sql.autoBroadcastJoinThreshold', '10MB')
joined = readings.join(F.broadcast(meters), 'meter_id').groupBy('city').agg(F.sum('kwh').alias('kwh'))
joined.explain(mode='formatted')
joined.show()

## 3. Cache y reutilización

In [ ]:
daily = readings.where('kwh >= 0').groupBy(F.to_date('ts').alias('day')).agg(F.sum('kwh').alias('kwh')).cache()
daily.count()  # primera acción materializa cache
daily.orderBy('day').show(5)
daily.unpersist()

## 4. Delta Lake
Crea una tabla independiente en `data/lake/sandbox`; cada write agrega un commit al transaction log.

In [ ]:
from delta.tables import DeltaTable
from lab.config import DATA

path = f'file://{DATA}/lake/sandbox/readings'
readings.limit(10000).write.format('delta').mode('overwrite').save(path)
DeltaTable.forPath(spark, path).history().select('version', 'operation').show()

## 5. Liberar recursos

In [ ]:
spark.stop()